# **Mapping Residential Development Trajectories as Indicators of Emerging and Future Business Markets**

Week 7

•	Emerging Market Score Analysis

## **Loading Data**

In [ ]:
import pandas as pd

# Loading the merged dataset
df_merged_final = pd.read_csv('clustered_df_merged_final.csv')

# Displaying the head and info of the loaded DataFrame
print("Head of the loaded merged dataset:")
print(df_merged_final.head())
print("\nInfo of the loaded merged dataset:")
df_merged_final.info()

Head of the loaded merged dataset:
     CSA     CBSA                                               Name  year  \
0  101.0  10180.0  Abilene, TX                                   ...  2024   
1  101.0  10180.0  Abilene, TX                                   ...  2024   
2  101.0  10180.0  Abilene, TX                                   ...  2024   
3  101.0  10180.0  Abilene, TX                                   ...  2024   
4  101.0  10180.0  Abilene, TX                                   ...  2024   

   month        date residential_structure_type  units_count  value_usd State  \
0      1  2024-01-01                     1 Unit         28.0       7967    TX   
1      2  2024-02-01                     1 Unit         31.0       8617    TX   
2      3  2024-03-01                     1 Unit         18.0       5107    TX   
3      4  2024-04-01                     1 Unit         43.0      11923    TX   
4      5  2024-05-01                     1 Unit         33.0       9490    TX   

   units_

In [ ]:
# Loading the Hotspot Data
hotspot_data = pd.read_csv('hotspot_cbsa_geo_data.csv')

# Displaying the head and info of the loaded DataFrame
print("Head of the loaded hotspot data:")
print(hotspot_data.head())
print("\nInfo of the loaded hotspot data:")
hotspot_data.info()

Head of the loaded hotspot data:
   CSAFP  CBSAFP        AFFGEOID  GEOID                                  NAME  \
0    NaN   43620  310M600US43620  43620                       Sioux Falls, SD   
1  174.0   16860  310M600US16860  16860                    Chattanooga, TN-GA   
2  324.0   29340  310M600US29340  29340                      Lake Charles, LA   
3  406.0   25220  310M600US25220  25220                           Hammond, LA   
4  122.0   12060  310M600US12060  12060  Atlanta-Sandy Springs-Alpharetta, GA   

                                          NAMELSAD LSAD        ALAND  \
0                       Sioux Falls, SD Metro Area   M1   6670256887   
1                    Chattanooga, TN-GA Metro Area   M1   5409949983   
2                      Lake Charles, LA Metro Area   M1   6083163521   
3                           Hammond, LA Metro Area   M1   2049124442   
4  Atlanta-Sandy Springs-Alpharetta, GA Metro Area   M1  22495780629   

       AWATER                                  

## **Emerging Market Score**

## **Feature Engineering: Defining the Emerging Market Score (EMS)**

To measure the scale, growth, and development activity of residential markets, the feature constructed is the composite Emerging Market Score (EMS) based on three dimensions of housing-market performance: market scale, growth momentum, and development intensity. These dimensions are motivated by recent empirical research examining regional housing-market levels, housing-market growth, and housing-supply responses (Anenberg & Ringo, 2024; Baum-Snow & Han, 2024; Pionnier & Schuffels, 2024).


#### **Market Scale (30%): Normalized units_count and value_usd**

Rationale: This dimension captures the underlying scale of residential-market activity through the number of housing units and the associated transaction value. Recent research emphasizes the importance of measuring regional housing-market levels using information on dwelling stocks and transaction prices when comparing housing markets across locations (Pionnier & Schuffels, 2024).

### **Growth Momentum (40%): Normalized units_growth_12m and value_growth_12m**

Rationale: This dimension captures the rate at which market activity is expanding. Recent evidence shows that fluctuations in housing demand are closely associated with changes in home sales and price growth, supporting the use of recent changes in transaction volumes and values as indicators of housing-market momentum (Anenberg & Ringo, 2024).

### **Development Intensity (30%): Normalized development_intensity and units_acceleration_1m**

Rationale: This dimension captures the supply-side response to housing-market conditions. Recent research demonstrates that housing-unit supply and new construction vary systematically across locations and that geographic characteristics influence the responsiveness of housing supply (Baum-Snow & Han, 2024).

The feature engineering involved robust min-max scaling, with extreme observations clipped at the 99th percentile, so that each component is comparable across markets and the resulting EMS ranges from 0 to 100. The 30%/40%/30% weighting scheme is a study-specific design choice intended to give greater emphasis to recent market growth while retaining substantial contributions from market scale and development activity.

Anenberg, E., & Ringo, D. (2024). Volatility in home sales and prices: Supply or demand? Journal of Urban Economics, 139, 103610. https://doi.org/10.1016/j.jue.2023.103610

Baum-Snow, N., & Han, L. (2024). The microgeography of housing supply. Journal of Political Economy, 132(6). https://doi.org/10.1086/728110

Pionnier, P.-A., & Schuffels, J. (2024). Estimating regional house price levels. Review of Income and Wealth, 70, 1116–1141. https://doi.org/10.1111/roiw.12676

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Making a copy of the dataset to avoid setting-with-copy warnings
df_ems = df_merged_final.copy()

# Core columns
core_cols = [
    'units_count', 'value_usd',
    'units_growth_12m', 'value_growth_12m',
    'development_intensity', 'units_acceleration_1m'
]

# Filling missing values in growth/acceleration/intensity columns with 0 or group medians for robust scoring
for col in core_cols:
    if col in ['units_growth_12m', 'value_growth_12m', 'units_acceleration_1m']:
        df_ems[col] = df_ems[col].fillna(0)
    elif col == 'development_intensity':
        df_ems[col] = df_ems[col].fillna(df_ems[col].median())

# Creating normalized columns clipped at 99th percentile to handle extreme real-estate outliers robustly
for col in core_cols:
    p99 = df_ems[col].quantile(0.99)
    clipped_series = df_ems[col].clip(upper=p99)
    # Min-Max Normalization
    min_val = clipped_series.min()
    max_val = clipped_series.max()
    if max_val - min_val > 0:
        df_ems[f'{col}_norm'] = (clipped_series - min_val) / (max_val - min_val)
    else:
        df_ems[f'{col}_norm'] = 0

# Calculating sub-scores
market_scale = (df_ems['units_count_norm'] + df_ems['value_usd_norm']) / 2
growth_momentum = (df_ems['units_growth_12m_norm'] + df_ems['value_growth_12m_norm']) / 2
development_pace = (df_ems['development_intensity_norm'] + df_ems['units_acceleration_1m_norm']) / 2

# Composite score calculation (Weighted average scaled to 0-100)
df_ems['emerging_market_score'] = (market_scale * 0.30 + growth_momentum * 0.40 + development_pace * 0.30) * 100

# Saving the modified DataFrame back
df_merged_final = df_ems

# Showing summary statistics and top records by Emerging Market Score
print("Summary of Emerging Market Score:")
print(df_merged_final['emerging_market_score'].describe())

print(
    "\nTop 10 Emerging Markets for 2026 (by Score):"
)
display(df_merged_final[df_merged_final['year'] == 2026][['Name', 'year', 'month', 'units_count', 'development_intensity', 'emerging_market_score']]
        .sort_values(by='emerging_market_score', ascending=False).head(10))

Summary of Emerging Market Score:
count    186404.000000
mean         28.681417
std           7.281765
min           5.228897
25%          26.849809
50%          26.849809
75%          28.157223
max          91.867418
Name: emerging_market_score, dtype: float64

Top 10 Emerging Markets for 2026 (by Score):


,Name,year,month,units_count,development_intensity,emerging_market_score
103523,"San Francisco-Oakland-Fremont, CA ...",2026,7,787.0,346.757306,91.743723
103837,"San Jose-Sunnyvale-Santa Clara, CA ...",2026,5,1391.0,171.401869,89.834650
16003,"Champaign-Urbana, IL ...",2026,3,953.0,229.302204,89.816414
107369,"Seattle-Tacoma-Bellevue, WA ...",2026,5,1644.0,210.179440,88.971744
41365,"Green Bay, WI ...",2026,1,247.0,798.959514,86.297203
81838,"New York-Newark-Jersey City, NY-NJ ...",2026,2,4594.0,181.773400,85.574559
4203,"Asheville, NC ...",2026,3,961.0,171.260146,85.158109
64662,"Los Angeles-Long Beach-Anaheim, CA ...",2026,2,4563.0,128.927241,84.945712
141539,"Fayetteville-Springdale-Rogers, AR ...",2026,3,792.0,247.782828,82.990306
10230,"Boston-Cambridge-Newton, MA-NH ...",2026,6,1267.0,216.875296,81.558174


## **Comparing Emerging Market Status**

In [ ]:
# Filtering for the year 2026 and get the top 20 by emerging market score
top_20_2026 = (df_merged_final[df_merged_final['year'] == 2026]
               .sort_values(by='emerging_market_score', ascending=False)
               .head(20))

# MergING with hotspot data on CBSA
# Note: df_merged_final has 'CBSA' and hotspot_data has 'CBSA'
merged_comparison = pd.merge(
    top_20_2026,
    hotspot_data[['CBSA', 'hotspot_type']],
    on='CBSA',
    how='left'
)

# SelectING and display the relevant columns for comparison
comparison_table = merged_comparison[[
    'Name', 'year', 'month', 'emerging_market_score', 'Cluster', 'hotspot_type'
]].rename(columns={'hotspot_type': 'Hotspot Status'})

print("Top 20 Emerging Markets in 2026 with Hotspot Status:")
display(comparison_table)

Top 20 Emerging Markets in 2026 with Hotspot Status:


,Name,year,month,emerging_market_score,Cluster,Hotspot Status
0,"San Francisco-Oakland-Fremont, CA ...",2026,7,91.743723,0.0,Not Significant
1,"San Jose-Sunnyvale-Santa Clara, CA ...",2026,5,89.834650,3.0,Not Significant
2,"Champaign-Urbana, IL ...",2026,3,89.816414,3.0,Not Significant
3,"Seattle-Tacoma-Bellevue, WA ...",2026,5,88.971744,2.0,Not Significant
4,"Green Bay, WI ...",2026,1,86.297203,2.0,Not Significant
5,"New York-Newark-Jersey City, NY-NJ ...",2026,2,85.574559,1.0,Not Significant
6,"Asheville, NC ...",2026,3,85.158109,3.0,Not Significant
7,"Los Angeles-Long Beach-Anaheim, CA ...",2026,2,84.945712,1.0,Not Significant
8,"Fayetteville-Springdale-Rogers, AR ...",2026,3,82.990306,1.0,Not Significant
9,"Boston-Cambridge-Newton, MA-NH ...",2026,6,81.558174,0.0,Not Significant


## **Saving Data**

In [ ]:
# Merging df_merged_final with hotspot_data on CBSA to include the 'hotspot_type' feature
df_merged_final_with_hotspots = pd.merge(
    df_merged_final,
    hotspot_data[['CBSA', 'hotspot_type']],
    on='CBSA',
    how='left'
)

# Saving the final consolidated dataset to a CSV file
save_path = 'final_df_merged_with_hotspots.csv'
df_merged_final_with_hotspots.to_csv(save_path, index=False)

# Confirming merge results
print(f"Data successfully merged and saved to: {save_path}")
print(f"Shape of final dataset: {df_merged_final_with_hotspots.shape}")
print("\nFirst few rows with the merged hotspot_type column:")
display(df_merged_final_with_hotspots[['Name', 'year', 'month', 'emerging_market_score', 'Cluster', 'hotspot_type']].head())

Data successfully merged and saved to: final_df_merged_with_hotspots.csv
Shape of final dataset: (186404, 28)

First few rows with the merged hotspot_type column:


,Name,year,month,emerging_market_score,Cluster,hotspot_type
0,"Abilene, TX ...",2024,1,29.889794,0.0,Not Significant
1,"Abilene, TX ...",2024,2,29.792200,0.0,Not Significant
2,"Abilene, TX ...",2024,3,29.497353,0.0,Not Significant
3,"Abilene, TX ...",2024,4,30.283091,0.0,Not Significant
4,"Abilene, TX ...",2024,5,30.046209,0.0,Not Significant
